In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.6.1


In [2]:
!python -V

Python 3.12.10


In [3]:
import pickle
import pandas as pd

In [4]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [7]:
year=2023
month=3
df = read_data(f'yellow_tripdata_{year}-{month:02d}.parquet')

In [9]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)
print(y_pred.std())

6.247488852238703


In [12]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
df['predicted_duration']=y_pred 
df_result=df.loc[:,['ride_id', 'predicted_duration']]

In [14]:
from pathlib import Path

# Build and create the directory
output_folder = Path("./output")
output_folder.mkdir(parents=True, exist_ok=True)  # parents=True is harmless here

# Construct the full path in an object-oriented way
output_file = output_folder / "homework_preds.parquet"

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [15]:
size_in_bytes = output_file.stat().st_size
print(f"File size: {size_in_bytes} bytes")

File size: 68640926 bytes
